In [1]:
import numpy as np
import polars as pl
from sklearn.svm import SVR, SVC
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import VarianceThreshold, SelectKBest, RFE, f_classif, r_regression, f_regression
from statsmodels.stats.outliers_influence import variance_inflation_factor

np.random.seed(13)

In [2]:
d = pl.DataFrame(
    np.array([
            [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
            [0, 1.1, 1.9, 3, 4, 5.1, 6, 7, 7.9, 9, 9.9],
            [0, 0.97, 2, 3, 4.1, 4.9, 6, 7.2, 8.1, 9.1, 10.1],
            [1, 1, 1, 1.1, 1, 1, 1, 0.9, 1, 1, 1.2],
            [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    ]), #+ np.random.rand(5, 11) / 10, 
    orient='col', 
    schema=['x', 'y', 'z', 'a', 't'],
)
X = d.select(pl.exclude('t'))
y = d['t'].cast(pl.Int32)
print(X)

n_features_to_select = 2
ml_type = 'regression'

shape: (11, 4)
┌──────┬─────┬──────┬─────┐
│ x    ┆ y   ┆ z    ┆ a   │
│ ---  ┆ --- ┆ ---  ┆ --- │
│ f64  ┆ f64 ┆ f64  ┆ f64 │
╞══════╪═════╪══════╪═════╡
│ 0.0  ┆ 0.0 ┆ 0.0  ┆ 1.0 │
│ 1.0  ┆ 1.1 ┆ 0.97 ┆ 1.0 │
│ 2.0  ┆ 1.9 ┆ 2.0  ┆ 1.0 │
│ 3.0  ┆ 3.0 ┆ 3.0  ┆ 1.1 │
│ 4.0  ┆ 4.0 ┆ 4.1  ┆ 1.0 │
│ …    ┆ …   ┆ …    ┆ …   │
│ 6.0  ┆ 6.0 ┆ 6.0  ┆ 1.0 │
│ 7.0  ┆ 7.0 ┆ 7.2  ┆ 0.9 │
│ 8.0  ┆ 7.9 ┆ 8.1  ┆ 1.0 │
│ 9.0  ┆ 9.0 ┆ 9.1  ┆ 1.0 │
│ 10.0 ┆ 9.9 ┆ 10.1 ┆ 1.2 │
└──────┴─────┴──────┴─────┘


In [3]:
print(X.corr())

shape: (4, 4)
┌──────────┬──────────┬──────────┬──────────┐
│ x        ┆ y        ┆ z        ┆ a        │
│ ---      ┆ ---      ┆ ---      ┆ ---      │
│ f64      ┆ f64      ┆ f64      ┆ f64      │
╞══════════╪══════════╪══════════╪══════════╡
│ 1.0      ┆ 0.999807 ┆ 0.999793 ┆ 0.240966 │
│ 0.999807 ┆ 1.0      ┆ 0.99946  ┆ 0.235546 │
│ 0.999793 ┆ 0.99946  ┆ 1.0      ┆ 0.234038 │
│ 0.240966 ┆ 0.235546 ┆ 0.234038 ┆ 1.0      │
└──────────┴──────────┴──────────┴──────────┘


In [4]:
# Variance Threshold
selector = VarianceThreshold(threshold=0.0)
X_ = selector.fit_transform(X)
print(selector.get_support())
print(np.array(X.columns)[selector.get_support()])

[ True  True  True  True]
['x' 'y' 'z' 'a']


In [5]:
# faster variance threshold
def drop_constant_columns(df: pl.DataFrame) -> pl.DataFrame:
    return df.select([
        col for col in df.columns
        if df[col].n_unique() > 1
    ])
X_ = drop_constant_columns(X)
print(X_.columns)

['x', 'y', 'z', 'a']


In [6]:
# faster variance threshold
def drop_low_variance_columns(df: pl.DataFrame, threshold: float = 0.0) -> pl.DataFrame:
    stats = df.select([
        pl.var(col).alias(col) for col in df.columns
    ])
    variances = stats.row(0)  # get variances as a list
    return df.select([
        col for col, var in zip(df.columns, variances) if var > threshold
    ])
X_ = drop_low_variance_columns(X)
print(X_.columns)    

['x', 'y', 'z', 'a']


In [7]:
# RFE
model = LinearRegression()
rfe = RFE(model, n_features_to_select=2)
rfe.fit(X, y)

# Get indices of selected features
selected_indices = rfe.get_support()
print('lrm indices:', selected_indices)

lrm indices: [ True  True False False]


In [8]:
if ml_type == 'classif':
    model = SVC(kernel='linear')
else:
    model = SVR(kernel='linear')
rfe = RFE(model, n_features_to_select=n_features_to_select)
rfe.fit(X, y)

# Get indices of selected features
selected_indices = rfe.get_support()
print("rfe indices:", selected_indices)

rfe indices: [ True False  True False]


In [9]:
selector = SelectKBest(
    f_regression,
    k=n_features_to_select,
).fit(X, y)
selected_indices = selector.get_support()
print("psn indices:", selected_indices)
print("psn  scores:", selector.scores_)

psn indices: [ True  True False False]
psn  scores: [1.79769313e+308 2.33388431e+004 2.17072982e+004 5.54794521e-001]


In [10]:
features = X
f = [
    variance_inflation_factor(features, feature_index)
    for feature_index in range(X.shape[1])
]
f

[27028.427329593127, 10233.722012637667, 9768.303129994598, 4.048631432232558]

In [11]:
# super slow
from sklearn.feature_selection import mutual_info_regression, mutual_info_classif
if ml_type == 'classif':
    mi_scores = mutual_info_classif(X, y, n_jobs=-1, discrete_features=False, random_state=13)
else:
    mi_scores = mutual_info_regression(X, y, n_jobs=-1, discrete_features=False, random_state=1)
mi_scores

array([0.98957431, 0.98199856, 0.94411977, 0.        ])

In [12]:
import xgboost as xgb
# model = xgb.XGBClassifier(
#     objective='binary:logistic',  # Default for binary classification
#     eval_metric='logloss',        # Can also use 'auc' or others
# )
# model = xgb.XGBClassifier(
#     objective='multi:softprob',   # Outputs class probabilities
#     eval_metric='mlogloss',       # Multiclass log-loss
#     num_class=3,                  # Replace with actual number of classes
# )
model = xgb.XGBRegressor(
    objective='reg:squarederror',  # Default for regression, minimizes squared error
    eval_metric='rmse',            # Metric for evaluation
)
model.fit(X, y)
importance = model.feature_importances_
print(importance)

[9.9982852e-01 0.0000000e+00 0.0000000e+00 1.7151532e-04]


In [13]:
import lightgbm as lgb
train_data = lgb.Dataset(data=X.to_arrow(), label=y.to_arrow())
# for small number of features, reduce `num_leaves` to avoid warnings
# for small number of data points, reduce `min_data_in_leave` to avoid warnings
# warnings #1: [Warning] No further splits with positive gain, best gain: -inf
# warnings #2: There are no meaningful features which satisfy the provided configuration. 
# Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
# params = {
#     'objective': 'binary',
#     'metric': 'binary_logloss', # or 'auc' for AUC score
# }
# params = {
#     'objective': 'multiclass',
#     'metric': 'multi_logloss',  # or 'multi_error'
#     'num_class': 3,             # replace 3 with the actual number of classes
# }
params = {
    'objective': 'regression', 
    'metric': 'rmse', 
    'verbose': -1, # no info messages
    'num_leaves': 2, 
    'min_data_in_leaf': 2,
}
model = lgb.train(params, train_data, num_boost_round=100, )
importance = model.feature_importance(importance_type='gain')
print(importance)

[515.89357967   0.           0.          31.43684727]


In [14]:
# feature importance is super slow, lgb: 35x faster, xgb: 15x faster
from catboost import CatBoostClassifier, CatBoostRegressor
# model = CatBoostClassifier(verbose=None)
model = CatBoostRegressor(verbose=0)
model.fit(X.to_pandas(), y.to_pandas())
importance = model.get_feature_importance()
print(importance)

[32.62766244 32.80378639 30.33472991  4.23382126]


In [15]:
# Permutation Importance example
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

# Train a model, Random Forest is a great choice as it's a powerful tree-based model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X.to_pandas(), y.to_pandas())

# Calculate Permutation Importance on the test set to get a more reliable estimate
perm = permutation_importance(model, X.to_pandas(), y.to_pandas(), n_repeats=10, random_state=13)

# Get and sort the `perm.importances_mean` attribute that gives the average importance score
df = pl.DataFrame({
    'feature': X.columns,
    'perm_score': perm.importances_mean,
    'perm_score_std': perm.importances_std,
}).sort('perm_score', descending=True)
print(df)

shape: (4, 3)
┌─────────┬────────────┬────────────────┐
│ feature ┆ perm_score ┆ perm_score_std │
│ ---     ┆ ---        ┆ ---            │
│ str     ┆ f64        ┆ f64            │
╞═════════╪════════════╪════════════════╡
│ y       ┆ 0.290504   ┆ 0.104094       │
│ x       ┆ 0.241296   ┆ 0.086679       │
│ z       ┆ 0.204661   ┆ 0.072679       │
│ a       ┆ 0.00252    ┆ 0.000983       │
└─────────┴────────────┴────────────────┘


In [16]:
# embedded Lasso (L1) regularization
from sklearn.linear_model import Lasso
# Initialize and train the Lasso model
model = Lasso(alpha=0.01)
model.fit(X, y)
# Features with a coefficient of 0 are not important
print(model.coef_)

[9.98985492e-01 0.00000000e+00 2.85879907e-05 0.00000000e+00]
